> **Archived run (24,566-row, 384x192 pipeline).** The stored outputs below are a
> record of that Kaggle session. Some cells use flags or modules that no longer exist
> (`--balance-datasets`, `--augment-strength`, `--select-metric`,
> `training.evaluate_per_dataset`, `--group-depth`). For the current 20,548-row,
> 256x512 pipeline use `docs/kaggle_stage3.md` instead.

# Signature verification - training and evaluation (CEDAR + BHSig260 + SigResize)

Trains the Stage 3 ConvNeXt-Tiny metric-learning encoder and evaluates it, on a Kaggle GPU.

**Prerequisite.** The attached dataset must have been preprocessed with the current Stage 2.2
path: pure-white background, true grayscale ink preserved, `256 x 512` canvas. Build it
locally with:

```bash
python -m preprocessing.build_manifests
python -m preprocessing.preprocess_dataset \\
  --manifest data/manifest_three.csv --image-root data \\
  --output-dir data/grayscale_processed --grayscale
```

then zip `data/grayscale_processed/` **flat** (`manifest.csv` at the archive root, image
folders beside it) and upload it as a private Kaggle dataset.

**What runs here**

1. environment + dataset checks
2. Stage 3 training **with augmentation** (`--augment`, training split only)
3. optional no-augmentation baseline for the ablation
4. fusion config fitted on the validation split
5. evaluation: deep vs fused, plus a per-dataset breakdown
6. artifact packaging


In [1]:
# Cell 1 - clone the repo and install only what Kaggle is missing.
# Kaggle images already ship torch/torchvision; reinstalling them wastes ~2 GB and time.
# Change REPO_URL if you are training from a different fork.
REPO_URL = "https://github.com/acharyayush/Signature-Verification-FIRST-.git"
!git clone $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
# Augmentation is pure OpenCV/NumPy, so albumentations is not needed.
!pip install -q --no-input timm opencv-python scikit-learn


Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 186, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 186 (delta 94), reused 134 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (186/186), 676.83 KiB | 5.83 MiB/s, done.
Resolving deltas: 100% (94/94), done.
/kaggle/working/repo


In [2]:
# Cell 2 - verify the checkout and the GPU.
import sys, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print(sys.version)


torch 2.11.0+cu130 | cuda True
gpu Tesla T4
3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [3]:
# Cell 3 - point at the attached preprocessed dataset.
# Use the notebook UI: Add Data -> Your Datasets -> attach the preprocessed dataset.
PROCESSED = "/kaggle/input/datasets/acharyayush2/384x192-grayscale-processed-3-dataset/grayscale_processed"
!ls -la {PROCESSED}
!head -3 {PROCESSED}/manifest.csv


total 1567
drwxrwxrwx 1 root root       0 Nov 24 04:29 BHSig260
drwxrwxrwx 1 root root       0 Nov 24 04:29 CEDAR
-rw-rw-rw- 1 root root 1603949 Nov 24 04:26 manifest.csv
drwxrwxrwx 1 root root       0 Nov 24 04:29 SigResize





In [4]:
# Cell 4 - verify counts and that every relative image path resolves.
import csv, collections
from pathlib import Path

root = Path(PROCESSED)
rows = list(csv.DictReader((root / "manifest.csv").open()))
print("rows    ", len(rows))
print("splits  ", collections.Counter(r["split"] for r in rows))
print("signers ", len({r["signer_id"] for r in rows}))
print("datasets", collections.Counter(r["image_path"].split("/")[0] for r in rows))
missing = [r["image_path"] for r in rows if not (root / r["image_path"]).exists()]
print("missing images", len(missing), missing[:3])
assert not missing, "dataset incomplete or manifest paths do not match the layout"


rows     24566
splits   Counter({'train': 17240, 'test': 3663, 'val': 3663})
signers  429
datasets Counter({'BHSig260': 14040, 'SigResize': 7934, 'CEDAR': 2592})
missing images 0 []


## 1. Train (with augmentation)

`--augment` applies tilt (up to 20 deg for `standard`), shear, mild perspective, elastic warp,
pen stroke width, ink contrast, blur and noise (gamma is off: it duplicates ink contrast) - to the **training split only**.
Validation is never augmented, because the checkpoint is selected by the validation EER.
Each augmented image is re-whitened and re-fitted through the same crop + resize as
inference, so the representation contract is unchanged.

`--balance-datasets` gives every source dataset an equal share of each batch, so BHSig260
(160 + 100 signers) does not crowd out CEDAR (55). `--select-metric eer` picks the checkpoint
by the reported metric rather than by the triplet-loss scale.

Expect roughly 25 epochs on a T4. Best weights land in `checkpoints/stage3_aug/best_model.pt`.
If you hit CUDA OOM, drop `--signers-per-batch` to 8.


In [5]:
# Cell 5 - Stage 3 training with augmentation (recommended run).
!python -m training.train_siamese \
  --manifest {PROCESSED}/manifest.csv \
  --already-preprocessed \
  --output-dir checkpoints/stage3_aug \
  --device cuda \
  --phase1-epochs 5 \
  --phase2-epochs 20 \
  --signers-per-batch 12 \
  --genuine-per-signer 4 \
  --forgeries-per-signer 2 \
  --augment \
  --augment-strength standard \
  --balance-datasets \
  --select-metric eer \
  --workers 2 \
  --seed 42


Dataset-balanced sampling over 3 datasets: [('BHSig260', 221), ('CEDAR', 46), ('SigResize', 98)]
Augmentation: strength=standard
Using device: cuda

model.safetensors: downloading bytes: █████▊                | 30.3MB, 1.03MB/s  
model.safetensors: downloading bytes: █████████▎            | 48.3MB, 2.98MB/s  
model.safetensors: downloading bytes: ███████████████████▉  |  104MB, 7.14MB/s  
model.safetensors: downloading bytes: ██████████████████████|  109MB, 10.4MB/s  
model.safetensors: reconstructing file: 100%|██████|  114MB /  114MB, 11.0MB/s  
{"phase": "warmup", "epoch": 1, "train": {"loss": 0.28640395402908325, "mean_positive": 0.7515550301625178, "mean_negative": 0.7766482188151433, "mean_identity_negative": 0.7766482188151433, "mean_forgery_negative": 0.7543087189014142, "forgery_coverage": 1.0, "active_fraction": 1.0}, "validation": {"loss": 0.20540167888005575, "mean_positive": 0.3561044782400131, "mean_negative": 0.4768134355545044, "mean_identity_negative": 0.47681343555450

## 2. Optional baseline (no augmentation)

Same settings without `--augment`, so the two runs differ only by augmentation. Compare the
`eer_skilled` values from Cell 8 and the per-dataset numbers from Cell 9 to see whether the
augmentation actually helped on your data.


In [6]:
# Cell 6 - baseline without augmentation (optional; skip to save GPU time).
!python -m training.train_siamese \
  --manifest {PROCESSED}/manifest.csv \
  --already-preprocessed \
  --output-dir checkpoints/stage3_base \
  --device cuda \
  --phase1-epochs 5 \
  --phase2-epochs 20 \
  --signers-per-batch 12 \
  --genuine-per-signer 4 \
  --forgeries-per-signer 2 \
  --balance-datasets \
  --select-metric eer \
  --workers 2 \
  --seed 42


Dataset-balanced sampling over 3 datasets: [('BHSig260', 221), ('CEDAR', 46), ('SigResize', 98)]
Using device: cuda
{"phase": "warmup", "epoch": 1, "train": {"loss": 0.24083774364911592, "mean_positive": 0.8019702182366297, "mean_negative": 0.8634390143247751, "mean_identity_negative": 0.8634390143247751, "mean_forgery_negative": 0.865813305744758, "forgery_coverage": 1.0, "active_fraction": 0.9967948725590339}, "validation": {"loss": 0.16130013515551886, "mean_positive": 0.48870359857877094, "mean_negative": 0.688113808631897, "mean_identity_negative": 0.688113808631897, "mean_forgery_negative": 0.6606965363025665, "forgery_coverage": 1.0, "active_fraction": 0.881944457689921}, "validation_eer_skilled": 0.2707231442339819}
{"phase": "warmup", "epoch": 2, "train": {"loss": 0.1863150676855674, "mean_positive": 0.773860555428725, "mean_negative": 0.8924802656357105, "mean_identity_negative": 0.8924802656357105, "mean_forgery_negative": 0.9195891893826998, "forgery_coverage": 1.0, "active

## 3. Fit the score-level fusion config

Fitted on the **validation split only**, with signer-grouped folds, so the stored threshold is
not the optimistic in-sample one. `--fusion-type both` fits the weighted and logistic rules and
keeps the lower out-of-fold EER. `--normalize-hand dataset` z-scores the absolute handcrafted
features per dataset, which matters for a multi-script manifest.


In [7]:
# Cell 7 - fit fusion on validation (uses the augmented checkpoint).
!python -m training.fit_fusion \
  --checkpoint checkpoints/stage3_aug/best_model.pt \
  --manifest {PROCESSED}/manifest.csv \
  --already-preprocessed \
  --references 5 \
  --device cuda \
  --batch-size 32 \
  --fusion-type both \
  --cv-folds 5 \
  --fit-unit query \
  --normalize-hand dataset \
  --output checkpoints/fusion/fusion.json


Read 3663 validation records
Embedded 3663 records
200/3663 signatures described
400/3663 signatures described
600/3663 signatures described
800/3663 signatures described
1000/3663 signatures described
1200/3663 signatures described
1400/3663 signatures described
1600/3663 signatures described
1800/3663 signatures described
2000/3663 signatures described
2200/3663 signatures described
2400/3663 signatures described
2600/3663 signatures described
2800/3663 signatures described
3000/3663 signatures described
3200/3663 signatures described
3400/3663 signatures described
3600/3663 signatures described
Described 3663 records
Built 83164 queries
Dataset labels: ['BHSig260', 'CEDAR', 'SigResize']
Built 415820 fitting pairs across 64 signers (6335 genuine)
Standardised handcrafted features per dataset: ['BHSig260', 'CEDAR', 'SigResize']
Fitted weighted fusion on 83164 query-level rows
weighted: out-of-fold EER 0.1388 at threshold 0.8587
note: --fit-unit query does not apply to the logistic rul

## 4. Evaluate on the test split

Cell 8 prints the deep baseline block and the fusion block side by side. The threshold comes
from the validation split / fusion config; test signers are never used for tuning.


In [8]:
# Cell 8 - deep vs fused, pooled over all test signers.
!python -m training.evaluate \
  --checkpoint checkpoints/stage3_aug/best_model.pt \
  --manifest {PROCESSED}/manifest.csv \
  --already-preprocessed \
  --references 5 \
  --device cuda \
  --batch-size 32 \
  --fusion-config checkpoints/fusion/fusion.json \
  --fusion-report checkpoints/fusion/report.json


applying per-dataset handcrafted standardisation from the fusion config (3 datasets)
200/3663 signatures described
400/3663 signatures described
600/3663 signatures described
800/3663 signatures described
1000/3663 signatures described
1200/3663 signatures described
1400/3663 signatures described
1600/3663 signatures described
1800/3663 signatures described
2000/3663 signatures described
2200/3663 signatures described
2400/3663 signatures described
2600/3663 signatures described
2800/3663 signatures described
3000/3663 signatures described
3200/3663 signatures described
3400/3663 signatures described
3600/3663 signatures described
200/3663 signatures described
400/3663 signatures described
600/3663 signatures described
800/3663 signatures described
1000/3663 signatures described
1200/3663 signatures described
1400/3663 signatures described
1600/3663 signatures described
1800/3663 signatures described
2000/3663 signatures described
2200/3663 signatures described
2400/3663 signatures des

Cell 9 breaks the same numbers down per source dataset (`--group-depth 1` groups by
`BHSig260` / `CEDAR` / `SigResize`). This is where you see whether augmentation helped CEDAR
(historically the weakest) without damaging BHSig260's connected Indic strokes.


In [9]:
# Cell 9 - per-dataset breakdown.
!python -m training.evaluate_per_dataset \
  --checkpoint checkpoints/stage3_aug/best_model.pt \
  --manifest {PROCESSED}/manifest.csv \
  --already-preprocessed \
  --references 5 \
  --device cuda \
  --batch-size 32 \
  --group-depth 1 \
  --fusion-config checkpoints/fusion/fusion.json \
  --report checkpoints/fusion/per_dataset.json


per-dataset handcrafted standardisation: ['BHSig260', 'CEDAR', 'SigResize']
val: 3663 records in 3 datasets: ['BHSig260', 'CEDAR', 'SigResize']
200/3663 signatures described
400/3663 signatures described
600/3663 signatures described
800/3663 signatures described
1000/3663 signatures described
1200/3663 signatures described
1400/3663 signatures described
1600/3663 signatures described
1800/3663 signatures described
2000/3663 signatures described
2200/3663 signatures described
2400/3663 signatures described
2600/3663 signatures described
2800/3663 signatures described
3000/3663 signatures described
3200/3663 signatures described
3400/3663 signatures described
3600/3663 signatures described
test: 3663 records in 3 datasets: ['BHSig260', 'CEDAR', 'SigResize']
200/3663 signatures described
400/3663 signatures described
600/3663 signatures described
800/3663 signatures described
1000/3663 signatures described
1200/3663 signatures described
1400/3663 signatures described
1600/3663 signatures

## 6. Package the artifacts and re-run the tests

`/kaggle/working` is what Kaggle saves as notebook output; `checkpoints/` is gitignored inside
the clone, so tarball what you want to keep.


In [10]:
# Cell 11 - package artifacts and confirm the repo is intact.
%cd /kaggle/working/repo
!ls -lh checkpoints/stage3_aug/best_model.pt checkpoints/fusion/fusion.json
!tar -czf /kaggle/working/artifacts.tar.gz \
  checkpoints/stage3_aug/best_model.pt \
  checkpoints/fusion/fusion.json \
  checkpoints/fusion/report.json \
  checkpoints/fusion/per_dataset.json
!ls -lh /kaggle/working/artifacts.tar.gz
!python -m unittest discover -s tests -t . 2>&1 | tail -5


/kaggle/working/repo
-rw-r--r-- 1 root root 4.3K Nov 24 05:27 checkpoints/fusion/fusion.json
-rw-r--r-- 1 root root 109M Nov 24 05:00 checkpoints/stage3_aug/best_model.pt
-rw-r--r-- 1 root root 101M Nov 24 05:39 /kaggle/working/artifacts.tar.gz
................................................................................................................
----------------------------------------------------------------------
Ran 160 tests in 3.880s

OK
